# Test notebook

The purpose of this notebook is to test an equation and compare them with the baselines: Burton, MBR, and DDM1, 2 and 3. We will also plot each storm and get the metrics for the equation.

The only cell that we have to modify is the following one, where we can change the features, the mode (template or default) and the output directory for the plots.
Raw EQ is the equation that we want to test, the raw version generated from the train_script.py file.

In [1]:
import os

FEATURES = ["P_dyn", "VBs", "epsilon", "DST"]
MODE = "template"  # 'template' or 'default'
OUTPUT_DIR = "comparison_template_deriv_against_baselines_review"
RAW_EQ = "g = (#2 * -0.0010713526) * sqrt(#1 + 1.32442); d = square((#1 * 0.01838707) - 0.5912093)"

output_folder = OUTPUT_DIR
# Count number of existing subfolders
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

import sympy as sp
from tqdm import tqdm

from sympy.printing import latex

# Internal module imports
import storm_dates
import baseline_models

# from evaluation_engine import UnifiedModel, simulate_storm, compute_features
from evaluation_engine import EquationModel, simulate_storm
from train_script import load_and_preprocess, compute_features

[juliapkg] Found dependencies: /mnt/data/symbolic-regression-dst-public-repo/.venv/lib/python3.12/site-packages/juliacall/juliapkg.json
[juliapkg] Found dependencies: /mnt/data/symbolic-regression-dst-public-repo/.venv/lib/python3.12/site-packages/pysr/juliapkg.json
[juliapkg] Found dependencies: /mnt/data/symbolic-regression-dst-public-repo/.venv/lib/python3.12/site-packages/juliapkg/juliapkg.json
[juliapkg] Locating Julia 1.10.3 - 1.11
[juliapkg] Querying Julia versions from https://julialang-s3.julialang.org/bin/versions.json
[juliapkg] Using Julia 1.11.9 at /mnt/data/symbolic-regression-dst-public-repo/.venv/julia_env/pyjuliapkg/install/bin/julia
[juliapkg] Using Julia project at /mnt/data/symbolic-regression-dst-public-repo/.venv/julia_env
[juliapkg] Writing Project.toml:
           | [deps]
           | PythonCall = "6099a3de-0909-46bc-b1f4-468b9a2dfc0d"
           | OpenSSL_jll = "458c3c95-2e84-50aa-8efc-19380b2a3a95"
           | SymbolicRegression = "8254be44-1295-4e6a-a16d-46

    Updating registry at `~/.julia/registries/General.toml`
   Resolving package versions...
    Updating `/mnt/data/symbolic-regression-ldi-internal/.venv/julia_env/Project.toml`
⌅ [6099a3de] + PythonCall v0.9.26
⌅ [8254be44] + SymbolicRegression v1.11.3
⌅ [458c3c95] + OpenSSL_jll v3.0.20+0
  [9e88b42a] ~ Serialization ⇒ v1.11.0
    Updating `/mnt/data/symbolic-regression-ldi-internal/.venv/julia_env/Manifest.toml`
  [47edcb42] + ADTypes v1.24.0
  [79e6a3ab] + Adapt v4.7.1
  [66dad0bd] + AliasTables v1.1.3
  [4fba245c] + ArrayInterface v7.30.2
  [d360d2e6] + ChainRulesCore v1.26.1
  [bbf7d656] + CommonSubexpressions v0.3.1
  [34da2185] + Compat v4.18.1
  [992eb4ea] + CondaPkg v0.2.36
  [187b0558] + ConstructionBase v1.6.0
  [9a962f9c] + DataAPI v1.16.0
  [864edb3b] + DataStructures v0.19.6
  [e2d170a0] + DataValueInterfaces v1.0.0
  [163ba53b] + DiffResults v1.1.0
  [b552c78f] + DiffRules v1.16.0
  [a0c0ee7d] + DifferentiationInterface v0.7.21
  [8d63f2c5] + DispatchDoctor v0.4.29
  [

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


/mnt/data/symbolic-regression-dst-public-repo/.venv/lib/python3.12/site-packages/spacepy/time.py:2448: UserWarning: Leapseconds may be out of date. Use spacepy.toolbox.update(leapsecs=True)
  _read_leaps()


In [3]:
raw_data = load_and_preprocess()
data = compute_features(raw_data)

In [4]:
def predict_and_plot_storm(model, start, end, storm_df, storm_id, save_path):
    # 1. Generate Predictions
    y_true = storm_df[start:end]["DST"].values

    res_eq = simulate_storm(model, storm_df)
    res_burton = baseline_models.burton_prediction(storm_df)
    res_obm = baseline_models.obm_prediction(storm_df)
    ddm1 = baseline_models.ddm1_prediction(storm_df)
    ddm2 = baseline_models.ddm2_prediction(storm_df)
    ddm3 = baseline_models.ddm3_prediction(storm_df)

    res_eq = res_eq[start:end]["DST_pred"].values
    res_burton = res_burton[start:end]["DST_pred"].values
    res_obm = res_obm[start:end]["DST_pred"].values
    res_ddm1 = ddm1[start:end]["DST_pred"].values
    res_ddm2 = ddm2[start:end]["DST_pred"].values
    res_ddm3 = ddm3[start:end]["DST_pred"].values

    # 2. Calculate Metrics
    m_eq = baseline_models.get_all_metrics(y_true, res_eq)
    m_burton = baseline_models.get_all_metrics(y_true, res_burton)
    m_obm = baseline_models.get_all_metrics(y_true, res_obm)
    m_ddm1 = baseline_models.get_all_metrics(y_true, res_ddm1)
    m_ddm2 = baseline_models.get_all_metrics(y_true, res_ddm2)
    m_ddm3 = baseline_models.get_all_metrics(y_true, res_ddm3)

    # 3. Setup Figure (3 Columns)
    fig, axs = plt.subplots(1, 3, figsize=(24, 11), constrained_layout=True)
    fig.suptitle(
        rf"Evaluation for Equation: ${model.latex_str()}$", fontsize=24, wrap=True
    )
    # Column 1: Time Series
    axs[0].plot(
        storm_df[start:end].index,
        y_true,
        color="black",
        label="Observed",
        alpha=0.6,
        linewidth=2,
    )
    axs[0].plot(
        storm_df[start:end].index,
        res_eq,
        color="blue",
        linestyle="--",
        label="Equation",
        linewidth=1.5,
    )
    axs[0].plot(
        storm_df[start:end].index,
        res_burton,
        color="yellow",
        linestyle="--",
        label="Burton",
        linewidth=1.5,
    )
    axs[0].plot(
        storm_df[start:end].index,
        res_obm,
        color="green",
        linestyle="--",
        label="OBM",
        linewidth=1.5,
    )
    
    axs[0].plot(
        storm_df[start:end].index,
        res_ddm1,
        color="orange",
        linestyle="--",
        label="DDM1",
        linewidth=1.5,
    )
    axs[0].plot(
        storm_df[start:end].index,
        res_ddm2,
        color="purple",
        linestyle="--",
        label="DDM2",
        linewidth=1.5,
    )
    axs[0].plot(
        storm_df[start:end].index,
        res_ddm3,
        color="cyan",
        linestyle="--",
        label="DDM3",
        linewidth=1.5,
    )
    
    axs[0].set_title(f"Storm {storm_id} Reconstruction", fontsize=24)
    axs[0].tick_params(axis='both', which='major', labelsize=20)
    axs[0].tick_params(axis='both', which='minor', labelsize=18)
    axs[0].legend(fontsize = 18)
    axs[0].grid(True)
    axs[0].set_xlim(start, end)
    axs[0].set_xlabel("Date", fontsize=20)
    axs[0].set_ylabel("Dst (nT)", fontsize=20)
    axs[0].xaxis.set_major_locator(MultipleLocator(2))

    # Column 2: Prediction Error
    diff_eq = res_eq - y_true
    diff_burton = res_burton - y_true
    diff_obm = res_obm - y_true
    diff_ddm1 = res_ddm1 - y_true
    diff_ddm2 = res_ddm2 - y_true
    diff_ddm3 = res_ddm3 - y_true


    axs[1].plot(storm_df[start:end].index, diff_eq, color="blue", label="Eq Error")
    axs[1].plot(
        storm_df[start:end].index, diff_burton, color="yellow", label="Burton Error"
    )
    axs[1].plot(storm_df[start:end].index, diff_obm, color="green", label="OBM Error")
    axs[1].plot(storm_df[start:end].index, diff_ddm1, color="orange", label="DDM1 Error")
    axs[1].plot(storm_df[start:end].index, diff_ddm2, color="purple", label="DDM2 Error")
    axs[1].plot(storm_df[start:end].index, diff_ddm3, color="cyan", label="DDM3 Error")
    axs[1].axhline(0, color="black", linestyle="--")
    axs[1].set_title("Prediction Error", fontsize=24)
    axs[1].set_ylabel("Error (nT)", fontsize=20)
    axs[1].set_xlabel("Date", fontsize=20)

    title_metrics = (
        f"Error Comparison\n"
        f"Eq: RMSE {m_eq[0]:.2f} | MAE {m_eq[1]:.2f} | R2 {m_eq[2]:.2f} | CC {m_eq[3]:.2f} | BFE {m_eq[4]:.2f}\n"
        f"Burton: RMSE {m_burton[0]:.2f} | MAE {m_burton[1]:.2f} | R2 {m_burton[2]:.2f} | CC {m_burton[3]:.2f} | BFE {m_burton[4]:.2f}\n"
        f"OBM: RMSE {m_obm[0]:.2f} | MAE {m_obm[1]:.2f} | R2 {m_obm[2]:.2f} | CC {m_obm[3]:.2f} | BFE {m_obm[4]:.2f}\n"
        f"DDM1: RMSE {m_ddm1[0]:.2f} | MAE {m_ddm1[1]:.2f} | R2 {m_ddm1[2]:.2f} | CC {m_ddm1[3]:.2f} | BFE {m_ddm1[4]:.2f}\n"
        f"DDM2: RMSE {m_ddm2[0]:.2f} | MAE {m_ddm2[1]:.2f} | R2 {m_ddm2[2]:.2f} | CC {m_ddm2[3]:.2f} | BFE {m_ddm2[4]:.2f}\n"
        f"DDM3: RMSE {m_ddm3[0]:.2f} | MAE {m_ddm3[1]:.2f} | R2 {m_ddm3[2]:.2f} | CC {m_ddm3[3]:.2f} | BFE {m_ddm3[4]:.2f}"
    )
    axs[1].set_title(title_metrics, fontsize=24)
    axs[1].set_ylabel("Error (nT)", fontsize=20)
    axs[1].set_xlabel("Date", fontsize=20)
    axs[1].legend(fontsize=18)
    axs[1].grid(True)
    axs[1].set_xlim(start, end)
    axs[1].tick_params(axis='both', which='major', labelsize=20)
    axs[1].tick_params(axis='both', which='minor', labelsize=18)
    
    axs[1].xaxis.set_major_locator(MultipleLocator(2))

    # Column 3: BFE
    baseline_models.plot_evaluation_bfe_multi(
        axs[2],
        y_true,
        [res_eq, res_burton, res_obm, res_ddm1, res_ddm2, res_ddm3],
        ["Equation", "Burton", "OBM", "DDM1", "DDM2", "DDM3"],
        ["blue", "yellow", "green", "orange", "purple", "cyan"],
        fontsize = 20,
        plot_legend=True
    )

    plt.savefig(save_path)
    plt.close()


In [5]:
def save_prediction_data(model, start, end, storm_df, output_path):
    """
    Generates and saves a CSV with observed and predicted DST and dDST/dt.
    """
    # 1. Observed Data
    # Real dDST is calculated as the difference to the next hour
    real_dst = storm_df[start:end]["DST"].values
    real_ddst = storm_df[start:end]["DST"].diff().shift(-1).values

    # 2. Equation Predictions
    # We need the iterative predictions for DST
    pred_dst_eq = simulate_storm(model, storm_df)
    if model.is_template:
        pred_dst_eq = pred_dst_eq[start:end][
            ["DST_pred", "dDST", "injection_component", "decay_component"]
        ]
    else:
        pred_dst_eq = pred_dst_eq[start:end][["DST_pred", "dDST"]]
    # 3. Baseline Predictions (Burton & OBM)
    pred_dst_burton = baseline_models.burton_prediction(storm_df)
    pred_dst_burton = pred_dst_burton[start:end][["DST_pred", "dDST"]]
    pred_dst_burton.columns = ["DST_pred_burton", "dDST_burton"]
    pred_dst_burton = pred_dst_burton[start:end][["DST_pred_burton", "dDST_burton"]]
    pred_dst_obm = baseline_models.obm_prediction(storm_df)
    pred_dst_obm = pred_dst_obm[start:end][["DST_pred", "dDST"]]
    pred_dst_obm.columns = ["DST_pred_obm", "dDST_obm"]
    pred_dst_obm = pred_dst_obm[start:end][["DST_pred_obm", "dDST_obm"]]
    pred_dst_ddm1 = baseline_models.ddm1_prediction(storm_df)    
    pred_dst_ddm1.columns = ["DST_pred_ddm1", "dDST_ddm1"]
    pred_dst_ddm1 = pred_dst_ddm1[start:end][["DST_pred_ddm1", "dDST_ddm1"]]
    pred_dst_ddm2 = baseline_models.ddm2_prediction(storm_df)
    pred_dst_ddm2.columns = ["DST_pred_ddm2", "dDST_ddm2"]
    pred_dst_ddm2 = pred_dst_ddm2[start:end][["DST_pred_ddm2", "dDST_ddm2"]]
    pred_dst_ddm3 = baseline_models.ddm3_prediction(storm_df)
    pred_dst_ddm3.columns = ["DST_pred_ddm3", "dDST_ddm3"]
    pred_dst_ddm3 = pred_dst_ddm3[start:end][["DST_pred_ddm3", "dDST_ddm3"]]


    # 4. Construct Comprehensive DataFrame

    if model.is_template:
        results_df = pd.DataFrame(
            {
                "Timestamp": storm_df[start:end].index,
                "Observed_DST": real_dst,
                "Real_dDST_dt": real_ddst,
                "Pred_DST_Equation": pred_dst_eq["DST_pred"].values,
                "Pred_dDST_dt_Equation": pred_dst_eq["dDST"].values,
                "Injection_Component": pred_dst_eq["injection_component"].values,
                "Decay_Component": pred_dst_eq["decay_component"].values,
                "Pred_DST_Burton": pred_dst_burton["DST_pred_burton"].values,
                "Pred_dDST_dt_Burton": pred_dst_burton["dDST_burton"].values,
                "Pred_DST_OBM": pred_dst_obm["DST_pred_obm"].values,
                "Pred_dDST_dt_OBM": pred_dst_obm["dDST_obm"].values,
                "Pred_DST_DDM1": pred_dst_ddm1["DST_pred_ddm1"].values,
                "Pred_dDST_dt_DDM1": pred_dst_ddm1["dDST_ddm1"].values,
                "Pred_DST_DDM2": pred_dst_ddm2["DST_pred_ddm2"].values,
                "Pred_dDST_dt_DDM2": pred_dst_ddm2["dDST_ddm2"].values,
                "Pred_DST_DDM3": pred_dst_ddm3["DST_pred_ddm3"].values,
                "Pred_dDST_dt_DDM3": pred_dst_ddm3["dDST_ddm3"].values,
            }
        ).set_index("Timestamp")
    else:
        results_df = pd.DataFrame(
            {
                "Timestamp": storm_df[start:end].index,
                "Observed_DST": real_dst,
                "Real_dDST_dt": real_ddst,
                "Pred_DST_Equation": pred_dst_eq["DST_pred"].values,
                "Pred_dDST_dt_Equation": pred_dst_eq["dDST"].values,
                "Pred_DST_Burton": pred_dst_burton["DST_pred_burton"].values,
                "Pred_dDST_dt_Burton": pred_dst_burton["dDST_burton"].values,
                "Pred_DST_OBM": pred_dst_obm["DST_pred_obm"].values,
                "Pred_dDST_dt_OBM": pred_dst_obm["dDST_obm"].values,
                "Pred_DST_DDM1": pred_dst_ddm1["DST_pred_ddm1"].values,
                "Pred_dDST_dt_DDM1": pred_dst_ddm1["dDST_ddm1"].values,
                "Pred_DST_DDM2": pred_dst_ddm2["DST_pred_ddm2"].values,
                "Pred_dDST_dt_DDM2": pred_dst_ddm2["dDST_ddm2"].values,
                "Pred_DST_DDM3": pred_dst_ddm3["DST_pred_ddm3"].values,
                "Pred_dDST_dt_DDM3": pred_dst_ddm3["dDST_ddm3"].values,

            }
        ).set_index("Timestamp")

    results_df.to_csv(output_path)
    return results_df

## Test storms

In [6]:
storms = []

model = EquationModel(RAW_EQ, FEATURES, is_template=MODE == "template")
test_storms = storm_dates.TEST_STORMS_SYMBOLIC_REGRESSION
storm_indices = []
for sd, ed, storm_id in tqdm(test_storms):
    start = pd.to_datetime(sd)
    end = pd.to_datetime(ed)
    storm_df = data[
        start - pd.DateOffset(hours=1) : end + pd.DateOffset(hours=1)
    ].copy()
    if storm_df.empty:
        continue

    file_name = f"storm_{storm_id}.png"
    predict_and_plot_storm(
        model,
        start,
        end,
        storm_df,
        storm_id,
        os.path.join(OUTPUT_DIR, file_name),
    )

    csv_name = f"data_storm_{storm_id}.csv"
    storms.append(
        save_prediction_data(
            model, start, end, storm_df, os.path.join(OUTPUT_DIR, csv_name)
        )
    )
    storm_indices.append(storm_id)
    
with open(os.path.join(OUTPUT_DIR, 'equation.txt'), 'w') as f:
    f.write(f'Equation: {RAW_EQ}\n')            
    f.write(f'LaTeX: {latex(model.latex_str())}\n')

  0%|          | 0/20 [00:00<?, ?it/s]

100%|██████████| 20/20 [00:12<00:00,  1.65it/s]


In [7]:
metrics = ["RMSE", "MAE", "R2", "CC", "BFE"]
equations = ["Equation", "Burton", "OBM", "DDM1", "DDM2", "DDM3"]

columns = [f"{eq}_{metric}" for eq in equations for metric in metrics]

summary_df = pd.DataFrame(
    columns=["Storm Index"] + columns,
)

for storm_index, storm in enumerate(storms):
    y_true = storm["Observed_DST"].values
    res_eq = storm["Pred_DST_Equation"].values
    res_burton = storm["Pred_DST_Burton"].values
    res_obm = storm["Pred_DST_OBM"].values
    res_ddm1 = storm["Pred_DST_DDM1"].values
    res_ddm2 = storm["Pred_DST_DDM2"].values
    res_ddm3 = storm["Pred_DST_DDM3"].values

    m_eq = baseline_models.get_all_metrics(y_true, res_eq)
    m_burton = baseline_models.get_all_metrics(y_true, res_burton)
    m_obm = baseline_models.get_all_metrics(y_true, res_obm)
    m_ddm1 = baseline_models.get_all_metrics(y_true, res_ddm1)
    m_ddm2 = baseline_models.get_all_metrics(y_true, res_ddm2)
    m_ddm3 = baseline_models.get_all_metrics(y_true, res_ddm3)

    summary_df.loc[len(summary_df)] = [
        storm_indices[storm_index],
        *m_eq,
        *m_burton,
        *m_obm,
        *m_ddm1,
        *m_ddm2,
        *m_ddm3,
    ]

summary_df.loc[len(summary_df)] = ["Mean", *summary_df[columns].mean().values]

global_data = pd.concat(storms, ignore_index=True)
y_true = global_data["Observed_DST"].values
res_eq = global_data["Pred_DST_Equation"].values
res_burton = global_data["Pred_DST_Burton"].values
res_obm = global_data["Pred_DST_OBM"].values
res_ddm1 = global_data["Pred_DST_DDM1"].values
res_ddm2 = global_data["Pred_DST_DDM2"].values
res_ddm3 = global_data["Pred_DST_DDM3"].values

m_eq = baseline_models.get_all_metrics(y_true, res_eq)
m_burton = baseline_models.get_all_metrics(y_true, res_burton)
m_obm = baseline_models.get_all_metrics(y_true, res_obm)
m_ddm1 = baseline_models.get_all_metrics(y_true, res_ddm1)
m_ddm2 = baseline_models.get_all_metrics(y_true, res_ddm2)
m_ddm3 = baseline_models.get_all_metrics(y_true, res_ddm3)

summary_df.loc[len(summary_df)] = [
    "Global",
    *m_eq,
    *m_burton,
    *m_obm,
    *m_ddm1,
    *m_ddm2,
    *m_ddm3,
]

display(summary_df)

,Storm Index,Equation_RMSE,Equation_MAE,Equation_R2,Equation_CC,Equation_BFE,Burton_RMSE,Burton_MAE,Burton_R2,Burton_CC,...,DDM2_RMSE,DDM2_MAE,DDM2_R2,DDM2_CC,DDM2_BFE,DDM3_RMSE,DDM3_MAE,DDM3_R2,DDM3_CC,DDM3_BFE
0,54.0,9.082450,7.150428,0.796405,0.922839,11.620567,22.373695,19.269475,-0.235483,0.821508,...,17.050674,14.716850,0.282463,0.811422,19.444409,17.429228,14.902320,0.250248,0.849179,20.944809
1,55.0,17.429458,13.783357,0.748936,0.886955,19.558619,22.579090,19.588765,0.578663,0.889269,...,20.994593,17.690741,0.635723,0.870003,22.069437,22.442717,18.173790,0.583737,0.864456,25.386413
2,56.0,13.276276,10.962537,0.613082,0.889956,15.586261,14.341538,9.959761,0.548500,0.860844,...,11.710892,9.329760,0.698945,0.874185,13.104050,11.359139,9.132990,0.716758,0.872736,12.821692
3,57.0,9.579700,7.665862,0.766690,0.899931,12.124595,19.246741,14.483019,0.058231,0.632441,...,8.231250,6.395718,0.827749,0.917543,11.379084,8.660890,6.313391,0.809298,0.904488,13.849705
4,58.0,9.165543,6.390252,0.797016,0.927846,15.584912,9.600592,7.210339,0.777290,0.887308,...,11.890757,9.533621,0.658364,0.924173,18.820727,11.862587,8.934864,0.659981,0.920502,19.970911
5,59.0,12.207318,10.241290,0.849938,0.955928,9.432197,16.064189,12.178801,0.740135,0.896003,...,10.840170,8.734074,0.881668,0.955437,13.497036,12.354960,9.578556,0.846286,0.954712,16.759531
6,60.0,17.617883,11.828793,0.777722,0.934481,26.807645,25.566310,20.034542,0.531914,0.929103,...,13.320289,10.330457,0.872938,0.963738,17.936865,14.721370,11.746767,0.844802,0.954256,19.773900
7,61.0,10.679534,8.252146,0.932006,0.968053,14.553100,40.570432,21.185698,0.018733,0.882420,...,22.490189,16.826801,0.698454,0.883757,30.616512,20.468681,15.214925,0.750226,0.886201,30.187252
8,62.0,11.394979,9.122644,0.698127,0.886644,12.820999,14.359693,10.613704,0.520612,0.816818,...,10.405858,7.936314,0.748260,0.888742,14.058279,11.805307,9.352887,0.675995,0.859124,16.881751
9,63.0,14.975924,12.167919,0.837054,0.944592,18.193426,22.708494,18.800912,0.625343,0.882998,...,19.780202,15.935485,0.715738,0.932886,27.615344,22.083822,17.372423,0.645672,0.929708,32.080943


In [8]:
display(summary_df.iloc[-2:, :].style.format({col: "{:.2f}" for col in columns}))

,Storm Index,Equation_RMSE,Equation_MAE,Equation_R2,Equation_CC,Equation_BFE,Burton_RMSE,Burton_MAE,Burton_R2,Burton_CC,Burton_BFE,OBM_RMSE,OBM_MAE,OBM_R2,OBM_CC,OBM_BFE,DDM1_RMSE,DDM1_MAE,DDM1_R2,DDM1_CC,DDM1_BFE,DDM2_RMSE,DDM2_MAE,DDM2_R2,DDM2_CC,DDM2_BFE,DDM3_RMSE,DDM3_MAE,DDM3_R2,DDM3_CC,DDM3_BFE
20,Mean,13.82,10.56,0.79,0.92,16.38,26.14,18.34,0.43,0.83,29.43,16.91,13.13,0.72,0.91,18.10,18.76,14.62,0.66,0.89,23.91,18.17,14.00,0.70,0.90,22.75,18.68,14.13,0.68,0.89,24.27
21,Global,14.01,10.25,0.89,0.94,28.79,28.71,16.83,0.52,0.86,78.08,17.34,12.71,0.82,0.92,44.10,19.88,13.56,0.77,0.89,38.92,19.09,13.14,0.79,0.90,45.73,19.56,13.39,0.78,0.89,46.40


In [9]:
display(summary_df[['Storm Index', 'Equation_BFE', 'Burton_BFE', 'OBM_BFE', 'DDM1_BFE', 'DDM2_BFE', 'DDM3_BFE']])

,Storm Index,Equation_BFE,Burton_BFE,OBM_BFE,DDM1_BFE,DDM2_BFE,DDM3_BFE
0,54.0,11.620567,22.860411,15.494723,21.129540,19.444409,20.944809
1,55.0,19.558619,19.865986,17.012856,23.435284,22.069437,25.386413
2,56.0,15.586261,20.706592,17.920288,13.908810,13.104050,12.821692
3,57.0,12.124595,15.793575,10.567192,13.078396,11.379084,13.849705
4,58.0,15.584912,11.977608,13.089744,20.108684,18.820727,19.970911
5,59.0,9.432197,16.351529,10.467582,14.982619,13.497036,16.759531
6,60.0,26.807645,33.653076,15.764727,36.458352,17.936865,19.773900
7,61.0,14.553100,59.661410,24.712335,28.911696,30.616512,30.187252
8,62.0,12.820999,13.397201,13.831007,14.844234,14.058279,16.881751
9,63.0,18.193426,24.687730,21.265740,29.938837,27.615344,32.080943


In [10]:
print(summary_df.loc[summary_df["Storm Index"].isin([57, 68, 'Mean'])].to_latex(index=False, float_format="%.3f"))

\begin{tabular}{lrrrrrrrrrrrrrrrrrrrrrrrrrrrrrr}
\toprule
Storm Index & Equation_RMSE & Equation_MAE & Equation_R2 & Equation_CC & Equation_BFE & Burton_RMSE & Burton_MAE & Burton_R2 & Burton_CC & Burton_BFE & OBM_RMSE & OBM_MAE & OBM_R2 & OBM_CC & OBM_BFE & DDM1_RMSE & DDM1_MAE & DDM1_R2 & DDM1_CC & DDM1_BFE & DDM2_RMSE & DDM2_MAE & DDM2_R2 & DDM2_CC & DDM2_BFE & DDM3_RMSE & DDM3_MAE & DDM3_R2 & DDM3_CC & DDM3_BFE \\
\midrule
57.000 & 9.580 & 7.666 & 0.767 & 0.900 & 12.125 & 19.247 & 14.483 & 0.058 & 0.632 & 15.794 & 10.407 & 8.641 & 0.725 & 0.856 & 10.567 & 8.754 & 6.721 & 0.805 & 0.921 & 13.078 & 8.231 & 6.396 & 0.828 & 0.918 & 11.379 & 8.661 & 6.313 & 0.809 & 0.904 & 13.850 \\
68.000 & 27.918 & 20.659 & 0.946 & 0.982 & 30.007 & 72.495 & 53.764 & 0.636 & 0.920 & 76.973 & 44.485 & 32.605 & 0.863 & 0.979 & 50.547 & 26.468 & 20.174 & 0.951 & 0.979 & 29.890 & 41.586 & 29.794 & 0.880 & 0.952 & 48.462 & 50.467 & 35.191 & 0.823 & 0.957 & 54.774 \\
Mean & 13.818 & 10.560 & 0.788 & 0.918 & 1

In [11]:
print(summary_df)

   Storm Index  Equation_RMSE  Equation_MAE  Equation_R2  Equation_CC  \
0         54.0       9.082450      7.150428     0.796405     0.922839   
1         55.0      17.429458     13.783357     0.748936     0.886955   
2         56.0      13.276276     10.962537     0.613082     0.889956   
3         57.0       9.579700      7.665862     0.766690     0.899931   
4         58.0       9.165543      6.390252     0.797016     0.927846   
5         59.0      12.207318     10.241290     0.849938     0.955928   
6         60.0      17.617883     11.828793     0.777722     0.934481   
7         61.0      10.679534      8.252146     0.932006     0.968053   
8         62.0      11.394979      9.122644     0.698127     0.886644   
9         63.0      14.975924     12.167919     0.837054     0.944592   
10        64.0      10.130700      8.022478     0.802369     0.916027   
11        65.0      14.249835     10.390993     0.633954     0.808397   
12        66.0      18.264381     15.416668     0.5

In [12]:
df = summary_df.copy()

In [13]:
# Keep only the 20 individual storms
storm_df = df[
    ~df["Storm Index"].isin(["Mean", "Global"])
].copy()

baselines = {
    "Burton": "Burton",
    "OBM": "OBM",
    "DDM#1": "DDM1",
    "DDM#2": "DDM2",
    "DDM#3": "DDM3",
}

metrics = {
    "RMSE": "lower",
    "MAE": "lower",
    "R2": "higher",
    "CC": "higher",
    "BFE": "lower",
}

rows = []

for metric, direction in metrics.items():

    row = {
        "Metric": metric,
        "Equation": storm_df[f"Equation_{metric}"].mean(),
    }

    for baseline_name, prefix in baselines.items():

        eq = storm_df[f"Equation_{metric}"]
        baseline = storm_df[f"{prefix}_{metric}"]

        row[baseline_name] = baseline.mean()

        if direction == "lower":
            wins = (eq < baseline).sum()
        else:
            wins = (eq > baseline).sum()

        row[f"Wins vs {baseline_name}"] = wins

    rows.append(row)

results_df = pd.DataFrame(rows)

print(results_df)

  Metric   Equation     Burton  Wins vs Burton        OBM  Wins vs OBM  \
0   RMSE  13.818406  26.143515              20  16.910134           17   
1    MAE  10.560487  18.335220              19  13.132016           17   
2     R2   0.788368   0.426411              20   0.717826           17   
3     CC   0.917989   0.829322              19   0.913725           12   
4    BFE  16.382973  29.429250              18  18.104329           12   

       DDM#1  Wins vs DDM#1      DDM#2  Wins vs DDM#2      DDM#3  \
0  18.760239             15  18.174985             13  18.675089   
1  14.616709             14  14.001200             14  14.130574   
2   0.660496             15   0.696041             13   0.682557   
3   0.894609             12   0.895495             15   0.893432   
4  23.906254             18  22.753257             16  24.270426   

   Wins vs DDM#3  
0             16  
1             14  
2             16  
3             16  
4             18  


In [14]:
latex_table = results_df.to_latex(
    index=False,
    escape=False,
    float_format="%.2f"
)

print(latex_table)

\begin{tabular}{lrrrrrrrrrrr}
\toprule
Metric & Equation & Burton & Wins vs Burton & OBM & Wins vs OBM & DDM#1 & Wins vs DDM#1 & DDM#2 & Wins vs DDM#2 & DDM#3 & Wins vs DDM#3 \\
\midrule
RMSE & 13.82 & 26.14 & 20 & 16.91 & 17 & 18.76 & 15 & 18.17 & 13 & 18.68 & 16 \\
MAE & 10.56 & 18.34 & 19 & 13.13 & 17 & 14.62 & 14 & 14.00 & 14 & 14.13 & 14 \\
R2 & 0.79 & 0.43 & 20 & 0.72 & 17 & 0.66 & 15 & 0.70 & 13 & 0.68 & 16 \\
CC & 0.92 & 0.83 & 19 & 0.91 & 12 & 0.89 & 12 & 0.90 & 15 & 0.89 & 16 \\
BFE & 16.38 & 29.43 & 18 & 18.10 & 12 & 23.91 & 18 & 22.75 & 16 & 24.27 & 18 \\
\bottomrule
\end{tabular}



In [15]:
results_df

,Metric,Equation,Burton,Wins vs Burton,OBM,Wins vs OBM,DDM#1,Wins vs DDM#1,DDM#2,Wins vs DDM#2,DDM#3,Wins vs DDM#3
0,RMSE,13.818406,26.143515,20,16.910134,17,18.760239,15,18.174985,13,18.675089,16
1,MAE,10.560487,18.335220,19,13.132016,17,14.616709,14,14.001200,14,14.130574,14
2,R2,0.788368,0.426411,20,0.717826,17,0.660496,15,0.696041,13,0.682557,16
3,CC,0.917989,0.829322,19,0.913725,12,0.894609,12,0.895495,15,0.893432,16
4,BFE,16.382973,29.429250,18,18.104329,12,23.906254,18,22.753257,16,24.270426,18


## Train storms

In [ ]:
storms = []

train_storms = storm_dates.TRAIN_STORMS_SYMBOLIC_REGRESSION
storm_indices = []
for sd, ed, storm_id in tqdm(train_storms):
    start = pd.to_datetime(sd)
    end = pd.to_datetime(ed)
    storm_df = data[
        start - pd.DateOffset(hours=1) : end + pd.DateOffset(hours=1)
    ].copy()
    if storm_df.empty:
        print(f'Storm {storm_id} has no data from {start} to {end}. Skipping.')
        continue

    file_name = f"storm_{storm_id}.png"
    predict_and_plot_storm(
        model,
        start,
        end,
        storm_df,
        storm_id,
        os.path.join(OUTPUT_DIR, file_name),
    )

    csv_name = f"data_storm_{storm_id}.csv"
    storms.append(
        save_prediction_data(
            model, start, end, storm_df, os.path.join(OUTPUT_DIR, csv_name)
        )
    )
    storm_indices.append(storm_id)

100%|██████████| 53/53 [01:15<00:00,  1.43s/it]


In [ ]:
metrics = ["RMSE", "MAE", "R2", "BFE"]
equations = ["Equation", "Burton", "OBM", "DDM1", "DDM2", "DDM3"]

columns = [f"{eq}_{metric}" for eq in equations for metric in metrics]

summary_df = pd.DataFrame(
    columns=["Storm Index"] + columns,
)

for storm_index, storm in enumerate(storms):
    y_true = storm["Observed_DST"].values
    res_eq = storm["Pred_DST_Equation"].values
    res_burton = storm["Pred_DST_Burton"].values
    res_obm = storm["Pred_DST_OBM"].values
    res_ddm1 = storm["Pred_DST_DDM1"].values
    res_ddm2 = storm["Pred_DST_DDM2"].values
    res_ddm3 = storm["Pred_DST_DDM3"].values

    m_eq = baseline_models.get_all_metrics(y_true, res_eq)
    m_burton = baseline_models.get_all_metrics(y_true, res_burton)
    m_obm = baseline_models.get_all_metrics(y_true, res_obm)
    m_ddm1 = baseline_models.get_all_metrics(y_true, res_ddm1)
    m_ddm2 = baseline_models.get_all_metrics(y_true, res_ddm2)
    m_ddm3 = baseline_models.get_all_metrics(y_true, res_ddm3)

    summary_df.loc[len(summary_df)] = [
        storm_indices[storm_index],
        *m_eq,
        *m_burton,
        *m_obm,
        *m_ddm1,
        *m_ddm2,
        *m_ddm3,
    ]

summary_df.loc[len(summary_df)] = ["Mean", *summary_df[columns].mean().values]

global_data = pd.concat(storms, ignore_index=True)
y_true = global_data["Observed_DST"].values
res_eq = global_data["Pred_DST_Equation"].values
res_burton = global_data["Pred_DST_Burton"].values
res_obm = global_data["Pred_DST_OBM"].values
res_ddm1 = global_data["Pred_DST_DDM1"].values
res_ddm2 = global_data["Pred_DST_DDM2"].values
res_ddm3 = global_data["Pred_DST_DDM3"].values

m_eq = baseline_models.get_all_metrics(y_true, res_eq)
m_burton = baseline_models.get_all_metrics(y_true, res_burton)
m_obm = baseline_models.get_all_metrics(y_true, res_obm)
m_ddm1 = baseline_models.get_all_metrics(y_true, res_ddm1)
m_ddm2 = baseline_models.get_all_metrics(y_true, res_ddm2)
m_ddm3 = baseline_models.get_all_metrics(y_true, res_ddm3)

summary_df.loc[len(summary_df)] = [
    "Global",
    *m_eq,
    *m_burton,
    *m_obm,
    *m_ddm1,
    *m_ddm2,
    *m_ddm3,
]

display(summary_df)

ValueError: cannot set a row with mismatched columns

In [ ]:
display(summary_df[['Storm Index', 'Equation_BFE', 'Burton_BFE', 'OBM_BFE', 'DDM1_BFE', 'DDM2_BFE', 'DDM3_BFE']])

,Storm Index,Equation_BFE,Burton_BFE,OBM_BFE,DDM1_BFE,DDM2_BFE,DDM3_BFE
0,1.0,16.822423,24.710095,10.111580,13.692198,15.087968,20.077620
1,2.0,19.405386,10.879065,12.884873,19.256149,18.623474,21.927198
2,3.0,8.793067,25.986482,17.414149,20.805647,19.258136,21.448388
3,4.0,19.953851,40.154114,18.849678,21.296023,22.493284,22.014097
4,5.0,24.272886,51.864800,23.564759,27.490935,29.206221,29.757253
5,6.0,16.965733,33.527777,14.085708,12.948621,17.609188,17.003694
6,7.0,7.050362,24.231238,13.020184,17.712507,14.937157,18.509792
7,8.0,13.263640,21.345670,10.522216,9.832557,8.673824,10.319187
8,9.0,11.592131,13.271597,7.515882,11.832959,10.296674,11.559362
9,10.0,16.813556,11.871591,12.972115,19.700706,23.927906,25.482754
